In [2]:
!pip install -q pandas numpy scikit-learn nltk spacy gensim matplotlib seaborn
!python -m spacy download pt_core_news_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/pt_core_news_sm-3.8.0/pt_core_news_sm-3.8.0-py3-none-any.whl (13.0 MB)
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import spacy

from nltk.corpus import stopwords
from gensim.models import FastText

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

nltk.download("stopwords", quiet=True)

stop_words_pt = set(stopwords.words("portuguese"))
nlp = spacy.load("pt_core_news_sm")

print("Ambiente preparado com sucesso.")

Ambiente preparado com sucesso.


In [7]:
from google.colab import files

uploaded = files.upload()

Saving dataset_sintetico_aula05.py to dataset_sintetico_aula05.py


In [8]:
import pandas as pd

# Executar o arquivo que você enviou
exec(open("/content/dataset_sintetico_aula05.py").read())

DATASETS DA MÓVEISDESIGN

Treinamento: 80 mensagens
Teste:       32 mensagens
OOD:         20 mensagens

Distribuição do treinamento:
intencao
logistica_entregas    20
trocas_devolucoes     20
vendas_orcamento      20
suporte_tecnico       20
Name: count, dtype: int64

Distribuição do teste:
intencao
vendas_orcamento      8
logistica_entregas    8
suporte_tecnico       8
trocas_devolucoes     8
Name: count, dtype: int64

Arquivos gerados:
✓ sac_moveis_ac2_treino.csv
✓ sac_moveis_ac2_teste.csv
✓ sac_moveis_ac2_ood.csv


In [10]:
df = pd.read_csv("sac_moveis_ac2_treino.csv")

print("Dimensões do dataset:", df.shape)
display(df.head())

Dimensões do dataset: (80, 2)


,mensagem,intencao
0,Meu pedido está atrasado,logistica_entregas
1,Quero devolver este sofá que chegou com rasgo,trocas_devolucoes
2,Qual o prazo de entrega do sofá que comprei?,logistica_entregas
3,Quero consultar o status da entrega,logistica_entregas
4,Recebi um armário com peças quebradas e quero ...,trocas_devolucoes


In [11]:
def limpar_e_lemmatizar(texto):
    """
    Realiza o pré-processamento de uma mensagem:

    1. Converte para minúsculas
    2. Remove caracteres especiais e números
    3. Processa o texto com spaCy
    4. Remove stopwords
    5. Realiza lemmatization
    6. Remove tokens muito pequenos
    7. Retorna o texto normalizado
    """

    # 1. Converter para minúsculas
    texto_limpo = texto.lower()

    # 2. Remover caracteres especiais e números
    texto_limpo = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_limpo
    )

    # 3. Processar com spaCy
    doc = nlp(texto_limpo)

    # 4. Extrair os lemas
    tokens_filtrados = []

    for token in doc:

        # Ignorar espaços
        if token.is_space:
            continue

        # Ignorar stopwords
        if token.text in stop_words_pt:
            continue

        # Ignorar tokens muito pequenos
        if len(token.text) <= 1:
            continue

        # Adicionar o lema
        tokens_filtrados.append(token.lemma_)

    # 5. Reconstruir a frase
    return " ".join(tokens_filtrados)

In [12]:
frase_teste = (
    "Gostaria de saber se vocês estão DEVOLVENDO "
    "os valores das mesas compradas!!!"
)

resultado = limpar_e_lemmatizar(frase_teste)

print("Frase original:")
print(frase_teste)

print("\nFrase processada:")
print(resultado)

Frase original:
Gostaria de saber se vocês estão DEVOLVENDO os valores das mesas compradas!!!

Frase processada:
gostar saber devolver valor meso comprada


In [13]:
def analisar_preprocessamento(texto):

    texto_normalizado = limpar_e_lemmatizar(texto)

    # Tokens antes do processamento
    texto_minusculo = texto.lower()
    texto_sem_especiais = re.sub(
        r'[^a-záàâãéèêíïóôõöúçñ\s]',
        '',
        texto_minusculo
    )

    doc = nlp(texto_sem_especiais)

    tokens_antes = [
        token.text
        for token in doc
        if not token.is_space
    ]

    tokens_finais = texto_normalizado.split()

    # Tokens removidos
    tokens_removidos = [
        token for token in tokens_antes
        if token not in tokens_finais
    ]

    print("Texto original:")
    print(texto)

    print("\nTexto normalizado:")
    print(texto_normalizado)

    print("\nQuantidade de caracteres:")
    print(len(texto))

    print("\nQuantidade de tokens antes:")
    print(len(tokens_antes))

    print("\nQuantidade de tokens depois:")
    print(len(tokens_finais))

    print("\nTokens removidos:")
    print(tokens_removidos)

    print("\nTokens finais:")
    print(tokens_finais)

In [14]:
analisar_preprocessamento(
    "MEU sofá!!! chegou quebrado e quero DEVOLVER!!!"
)

Texto original:
MEU sofá!!! chegou quebrado e quero DEVOLVER!!!

Texto normalizado:
sofá chegar quebrar querer devolver

Quantidade de caracteres:
47

Quantidade de tokens antes:
7

Quantidade de tokens depois:
5

Tokens removidos:
['meu', 'chegou', 'quebrado', 'e', 'quero']

Tokens finais:
['sofá', 'chegar', 'quebrar', 'querer', 'devolver']


In [15]:
df["texto_processado"] = df["mensagem"].apply(
    limpar_e_lemmatizar
)

df["tokens"] = df["texto_processado"].apply(
    lambda x: x.split()
)

display(df[["mensagem", "texto_processado", "tokens"]].head())

,mensagem,texto_processado,tokens
0,Meu pedido está atrasado,pedido atrasado,"[pedido, atrasado]"
1,Quero devolver este sofá que chegou com rasgo,querer devolver sofá chegar rasgo,"[querer, devolver, sofá, chegar, rasgo]"
2,Qual o prazo de entrega do sofá que comprei?,prazo entrega sofá comprei,"[prazo, entrega, sofá, comprei]"
3,Quero consultar o status da entrega,querer consultar status entrega,"[querer, consultar, status, entrega]"
4,Recebi um armário com peças quebradas e quero ...,recebi armário peça quebrar querer devolver,"[recebi, armário, peça, quebrar, querer, devol..."


In [16]:
corpus_tokenizado = df["tokens"].tolist()

print("Quantidade de mensagens no corpus:", len(corpus_tokenizado))
print("Exemplo:")
print(corpus_tokenizado[0])

Quantidade de mensagens no corpus: 80
Exemplo:
['pedido', 'atrasado']


In [17]:
modelo_fasttext = FastText(
    sentences=corpus_tokenizado,
    vector_size=50,
    window=3,
    min_count=1,
    workers=4,
    sg=1
)

print("Modelo FastText treinado com sucesso.")

Modelo FastText treinado com sucesso.


In [18]:
print(
    "Dimensão do vetor:",
    modelo_fasttext.wv.vector_size
)

Dimensão do vetor: 50


In [19]:
def obter_vetor_frase(frase, modelo):
    """
    Transforma uma frase em um vetor denso utilizando
    FastText + Mean Pooling.
    """

    palavras = frase.split()

    vetores = []

    for palavra in palavras:

        # Obter vetor da palavra
        vetor = modelo.wv[palavra]

        # Adicionar vetor à lista
        vetores.append(vetor)

    # Caso não existam vetores
    if len(vetores) == 0:
        return np.zeros(modelo.vector_size)

    # Mean Pooling
    vetor_medio = np.mean(vetores, axis=0)

    return vetor_medio

In [20]:
frase = "quero devolver meu sofá"

frase_processada = limpar_e_lemmatizar(frase)

vetor = obter_vetor_frase(
    frase_processada,
    modelo_fasttext
)

print("Frase processada:")
print(frase_processada)

print("\nDimensão do vetor:")
print(vetor.shape)

print("\nVetor:")
print(vetor)

Frase processada:
querer devolver sofá

Dimensão do vetor:
(50,)

Vetor:
[-1.1444055e-03  3.8392479e-03 -2.1939666e-03 -7.9391472e-04
 -1.4140267e-03  8.9081826e-05  4.1476716e-04  3.0109684e-03
  1.0967570e-03 -7.6226890e-04  3.0127533e-03  1.0160076e-03
 -8.5002044e-04  9.0500439e-04  1.9247766e-03  1.2719772e-03
  2.0829651e-03 -9.5745240e-04 -2.6341403e-04 -8.0457382e-04
 -6.1296619e-04 -3.3252973e-03  3.4528822e-03 -2.2243378e-03
  2.8998780e-04  8.1541244e-04 -1.5951727e-03 -6.4407213e-04
  4.4661476e-03 -2.4794692e-03 -1.9865241e-03 -5.3886889e-04
  1.0192011e-03  4.2558536e-03 -4.6039640e-05  1.5072023e-03
 -1.3652485e-04 -7.1902253e-04  1.1253973e-03 -2.9362978e-03
 -7.5033261e-04 -1.9030712e-03 -3.0656112e-05  2.5657561e-04
  3.4153965e-04 -3.4380525e-03  7.1787671e-04  2.2113265e-03
 -2.0023379e-03  1.6102478e-03]


In [21]:
X_vetores = np.array([
    obter_vetor_frase(texto, modelo_fasttext)
    for texto in df["texto_processado"]
])

y = df["intencao"].values

In [22]:
print("Formato de X:", X_vetores.shape)
print("Formato de y:", y.shape)

Formato de X: (80, 50)
Formato de y: (80,)


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X_vetores,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treinamento:", X_train.shape)
print("Teste:", X_test.shape)

Treinamento: (64, 50)
Teste: (16, 50)


In [24]:
modelo = LogisticRegression(
    max_iter=1000
)

modelo.fit(X_train, y_train)

print("Modelo treinado com sucesso.")

Modelo treinado com sucesso.


In [25]:
y_pred = modelo.predict(X_test)

print("=== RELATÓRIO DE CLASSIFICAÇÃO ===")
print(
    classification_report(
        y_test,
        y_pred
    )
)

=== RELATÓRIO DE CLASSIFICAÇÃO ===
                    precision    recall  f1-score   support

logistica_entregas       1.00      0.75      0.86         4
   suporte_tecnico       1.00      0.75      0.86         4
 trocas_devolucoes       0.67      1.00      0.80         4
  vendas_orcamento       0.50      0.50      0.50         4

          accuracy                           0.75        16
         macro avg       0.79      0.75      0.75        16
      weighted avg       0.79      0.75      0.75        16



In [26]:
def classificar_mensagem(
    mensagem,
    modelo,
    modelo_embedding,
    limiar=0.50
):

    # Pré-processamento
    mensagem_processada = limpar_e_lemmatizar(
        mensagem
    )

    # Gerar vetor
    vetor = obter_vetor_frase(
        mensagem_processada,
        modelo_embedding
    )

    vetor = vetor.reshape(1, -1)

    # Probabilidades
    probabilidades = modelo.predict_proba(vetor)

    # Maior probabilidade
    indice = np.argmax(probabilidades)

    confianca = probabilidades[0][indice]

    intencao = modelo.classes_[indice]

    # Fallback
    if confianca >= limiar:
        resposta = intencao
    else:
        resposta = "FALLBACK_HUMANO"

    return resposta, confianca

In [27]:
testes = [
    "quero devolver meu sofá",
    "como faço para realizar a devolução?",
    "cadê meu pedido?",
    "meu pedido nao chego",
    "qual é a previsão do tempo?"
]

for mensagem in testes:

    intencao, confianca = classificar_mensagem(
        mensagem,
        modelo,
        modelo_fasttext
    )

    print("=" * 60)
    print("Mensagem:", mensagem)
    print("Intenção:", intencao)
    print(
        "Confiança:",
        f"{confianca * 100:.2f}%"
    )

Mensagem: quero devolver meu sofá
Intenção: FALLBACK_HUMANO
Confiança: 25.00%
Mensagem: como faço para realizar a devolução?
Intenção: FALLBACK_HUMANO
Confiança: 25.00%
Mensagem: cadê meu pedido?
Intenção: FALLBACK_HUMANO
Confiança: 25.01%
Mensagem: meu pedido nao chego
Intenção: FALLBACK_HUMANO
Confiança: 25.01%
Mensagem: qual é a previsão do tempo?
Intenção: FALLBACK_HUMANO
Confiança: 25.00%


In [28]:
X_train_vec = X_train
X_test_vec = X_test

In [29]:
modelo_logistico = LogisticRegression(
    max_iter=1000
)

modelo_logistico.fit(
    X_train_vec,
    y_train
)

y_pred_logistico = modelo_logistico.predict(
    X_test_vec
)

In [30]:
modelo_knn = KNeighborsClassifier(
    n_neighbors=3
)

modelo_knn.fit(
    X_train_vec,
    y_train
)

y_pred_knn = modelo_knn.predict(
    X_test_vec
)

In [31]:
resultados = []

# Regressão Logística
resultados.append({
    "Modelo": "Regressão Logística",
    "Accuracy": accuracy_score(
        y_test,
        y_pred_logistico
    ),
    "Precision": precision_score(
        y_test,
        y_pred_logistico,
        average="weighted"
    ),
    "Recall": recall_score(
        y_test,
        y_pred_logistico,
        average="weighted"
    ),
    "F1": f1_score(
        y_test,
        y_pred_logistico,
        average="weighted"
    )
})

# KNN
resultados.append({
    "Modelo": "KNN",
    "Accuracy": accuracy_score(
        y_test,
        y_pred_knn
    ),
    "Precision": precision_score(
        y_test,
        y_pred_knn,
        average="weighted"
    ),
    "Recall": recall_score(
        y_test,
        y_pred_knn,
        average="weighted"
    ),
    "F1": f1_score(
        y_test,
        y_pred_knn,
        average="weighted"
    )
})

tabela_resultados = pd.DataFrame(resultados)

display(tabela_resultados)

,Modelo,Accuracy,Precision,Recall,F1
0,Regressão Logística,0.7500,0.791667,0.7500,0.753571
1,KNN,0.5625,0.525000,0.5625,0.521429


In [32]:
tabela_percentual = tabela_resultados.copy()

for coluna in [
    "Accuracy",
    "Precision",
    "Recall",
    "F1"
]:
    tabela_percentual[coluna] = (
        tabela_percentual[coluna] * 100
    ).round(2).astype(str) + "%"

display(tabela_percentual)

,Modelo,Accuracy,Precision,Recall,F1
0,Regressão Logística,75.0%,79.17%,75.0%,75.36%
1,KNN,56.25%,52.5%,56.25%,52.14%


In [33]:
tabela_resultados

,Modelo,Accuracy,Precision,Recall,F1
0,Regressão Logística,0.7500,0.791667,0.7500,0.753571
1,KNN,0.5625,0.525000,0.5625,0.521429
